[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1mk0QXjREp-k_J-pdFfmgoC7-EVmgPyAo#scrollTo=g4gyOZKkHDVU)

# ✔ Python libs: PyTorch and Cupy

Basics of CuPy
In this section, you will learn about the following things:

- Basics of cupy.ndarray

- The concept of current device

- Host-device and device-device array transfer



In [ ]:
# installation required on Colab on CUDA 10.2
!pip install cupy-cuda102


CuPy is a GPU array backend that implements a subset of NumPy interface. In the following code, `cp` is an abbreviation of `cupy`, following the convention of abbreviating `numpy` to `np`:

In [ ]:
import numpy as np
import cupy as cp

The `cupy.ndarray` class is in its core, which is a compatible GPU alternative of `numpy.ndarray`.

In [ ]:
x_gpu = cp.array([1, 2, 3])

`x_gpu` in the above example is an instance of `cupy.ndarray`. You can see its creation of identical to NumPy’s one, except that `numpy` is replaced with `cupy`. The main difference of `cupy.ndarray` from `numpy.ndarray` is that the content is allocated on the device memory. Its data is allocated on the current device, which will be explained later.

Most of the array manipulations are also done in the way similar to NumPy. Take the Euclidean norm (a.k.a L2 norm) for example. NumPy has `numpy.linalg.norm()` to calculate it on CPU.

In [ ]:
x_cpu = np.array([1, 2, 3])
l2_cpu = np.linalg.norm(x_cpu)
print(l2_cpu)

x_gpu = cp.array([1, 2, 3])
l2_gpu = cp.linalg.norm(x_gpu)
print(l2_gpu)

CuPy has a concept of current devices, which is the default device on which the allocation, manipulation, calculation, etc., of arrays are taken place. Suppose the ID of current device is 0. The following code allocates array contents on GPU 0.

In [ ]:
x_on_gpu0 = cp.array([1, 2, 3, 4, 5])
# cp.cuda.Device(1).use() # trigger dev 1

__Move arrays to a device__ 

`cupy.asarray()` can be used to move a `numpy.ndarray`, a `list`, or any `object` that can be passed to `numpy.array()` to the current device:

In [ ]:
x_cpu = np.array([1, 2, 3])
x_gpu = cp.asarray(x_cpu)  # move the data to the current device.

`cupy.asarray()` can accept `cupy.ndarray`, which means we can transfer the array between devices with this function.

In [ ]:
with cp.cuda.Device(0):
  x_gpu_0 = cp.ndarray([1, 2, 3])  # create an array in GPU 0
with cp.cuda.Device(1):
  x_gpu_1 = cp.asarray(x_gpu_0)  # move the array to GPU 1

__Move array from a device to the host__

Moving a device array to the host can be done by `cupy.asnumpy()` as follows:

In [ ]:
x_gpu = cp.array([1, 2, 3])  # create an array in the current device
x_cpu = cp.asnumpy(x_gpu)  # move the array to the host.

# We can also use cupy.ndarray.get():
x_cpu = x_gpu.get()

__How to write CPU/GPU agnostic code__

The compatibility of CuPy with NumPy enables us to write CPU/GPU generic code. It can be made easy by the `cupy.get_array_module()` function. This function returns the numpy or cupy module based on arguments. A CPU/GPU generic function is defined using it like follows:



In [ ]:
# Stable implementation of log(1 + exp(x))
def softplus(x):
  xp = cp.get_array_module(x)
  return xp.maximum(0, x) + xp.log1p(xp.exp(-abs(x)))

Sometimes, an explicit conversion to a host or device array may be required. `cupy.asarray()` and `cupy.asnumpy()` can be used in agnostic implementations to get host or device arrays from either CuPy or NumPy arrays.

In [ ]:
y_cpu = np.array([4, 5, 6])
x_cpu + y_cpu
x_gpu + y_cpu

In [ ]:
cp.asnumpy(x_gpu) + y_cpu
cp.asnumpy(x_gpu) + cp.asnumpy(y_cpu)
x_gpu + cp.asarray(y_cpu)
cp.asarray(x_gpu) + cp.asarray(y_cpu)

# sgemm with Cupy

In [ ]:
import os

def working_dir():
  # dir of the current lab
  src_dir = labs_base_dir + lab_dir

  # change working dir
  os.chdir(src_dir)

# set base dir of labs 
labs_base_dir = '/home/grossi/CUDA'
lab_dir = '/lab8/python'

# change working dir
working_dir()
print('Currente dir: ' + os.getcwd())

In [ ]:
%%writefile sgemm.cu

/*
Original works by:
--------------------------------------------------------
MAGMA
Copyright (c) 2017 The University of Tennessee. All rights reserved.
Licensed under modified BSD license
*/


// These parameters will be determined by utils.read_code
//#define DIM_X  16
//#define DIM_Y  16
//#define BLK_M  64
//#define BLK_N  64
//#define BLK_K  4
//#define DIM_XA  64
//#define DIM_YA  4
//#define DIM_XB  4
//#define DIM_YB  64
//#define THR_N  (BLK_N // DIM_Y)
//#define THR_M  (BLK_M // DIM_X)

#define fetch(arr, col, m, n, bound) arr[min(n*col + m, bound)]


extern "C" __global__
void sgemm(
        int M, int N, int K,
        const float* A,
        const float* B,
        float * C)
{
    int idx = threadIdx.x;
    int idy = threadIdx.y;

    int idt = DIM_X * idy + idx;

    int idxA = idt % DIM_XA;
    int idyA = idt / DIM_XA;

    int idxB = idt % DIM_XB;
    int idyB = idt / DIM_XB;

    int blx = blockIdx.x;
    int bly = blockIdx.y;

    __shared__ float sA[BLK_K][BLK_M + 1];
    __shared__ float sB[BLK_N][BLK_K + 1];

    // registers for the innermost loop
    float rC[THR_N][THR_M];
    float rA[THR_M];
    float rB[THR_N];

    float ra[BLK_K / DIM_YA][BLK_M / DIM_XA];
    float rb[BLK_N / DIM_YB][BLK_K / DIM_XB];

    const float* offs_dA = A + blx * BLK_M       + idyA * M + idxA;
    int boundA = (M * (K - 1) + M) - (blx * BLK_M + idyA * M + idxA) - 1;
    const float* offs_dB = B + bly * BLK_N * K + idyB * K + idxB;
    int boundB = (K * (N - 1) + K) - (bly * BLK_N * K + idyB * K + idxB) - 1;

    int m, n, k, kk;
    
    #pragma unroll
    for (n = 0; n < THR_N; n++) {
        #pragma unroll
        for (m = 0 ; m < THR_M; m++) {
            rC[n][m] = 0;
        }
    }

    // blockwise transpose to transpose load
    #pragma unroll
    for (n = 0; n < BLK_K; n += DIM_YA) {
        #pragma unroll
        for (m = 0; m < BLK_M; m += DIM_XA) {
            sA[n + idyA][m + idxA] = fetch(offs_dA, M, m, n, boundA);
        }
    }
    // blockwise transpose to transpose load
    #pragma unroll
    for (n = 0; n < BLK_N; n += DIM_YB) {
        #pragma unroll
        for (m = 0; m < BLK_K; m += DIM_XB) {
            sB[n + idyB][m + idxB] = fetch(offs_dB, K, m, n, boundB);
        }
    }
    __syncthreads();

    for (kk = 0; kk < K - BLK_K; kk += BLK_K)
    {
        offs_dA += BLK_K * M;
        boundA -= BLK_K * M;
        offs_dB += BLK_K;
        boundB -= BLK_K;
        
        #pragma unroll
        for (n = 0; n < BLK_K / DIM_YA; n++) {
            #pragma unroll
            for (m = 0; m < BLK_M / DIM_XA; m++) {
                ra[n][m] = fetch(offs_dA, M, m * DIM_XA, n * DIM_YA, boundA);
            }
        }

        #pragma unroll
        for (n = 0; n < BLK_N / DIM_YB; n++) {
            #pragma unroll
            for (m = 0; m < BLK_K / DIM_XB; m++) {
                rb[n][m] = fetch(offs_dB, K, m * DIM_XB, n * DIM_YB, boundB);
            }
        }

        // multiply
        #pragma unroll
        for (k = 0; k < BLK_K; k++)
        {
            #pragma unroll
            for (m = 0; m < THR_M; m++) {
                rA[m] = sA[k][m * DIM_X + idx];
            }
            
            #pragma unroll
            for (n = 0; n < THR_N; n++) {
                rB[n] = sB[n * DIM_Y + idy][k];
            }

            #pragma unroll
            for (n = 0; n < THR_N; n++) {
                #pragma unroll
                for (m = 0; m < THR_M; m++) {
                    rC[n][m] += rA[m] * rB[n];
                }
            }
        }
        __syncthreads();

        // store A regs->smem
        #pragma unroll
        for (n = 0; n < BLK_K / DIM_YA; n++)
        {
            #pragma unroll
            for (m = 0; m < BLK_M / DIM_XA; m++)
            {
                sA[n * DIM_YA + idyA][m * DIM_XA + idxA] = ra[n][m];
            }
        }

        #pragma unroll
        for (n = 0; n < BLK_N / DIM_YB; n++)
        {
            #pragma unroll
            for (m = 0; m < BLK_K / DIM_XB; m++)
            {
                sB[n * DIM_YB + idyB][m * DIM_XB + idxB] = rb[n][m];
            }
        }
        __syncthreads();
    }

    // Multiply last full (BLK_K) or partial block of columns of A and
    // rows of B.
    // It's okay that m,n exceed matrix bounds as all work is in registers
    // or shared memory, and out-of-bounds rC[n][m] will not be saved later.

    kk = K - kk;
    #pragma unroll
    for (k = 0; k < kk; k++)
    {
        #pragma unroll
        for (m = 0; m < THR_M; m++) {
            rA[m] = sA[k][m * DIM_X + idx];
        }

        #pragma unroll
        for (n = 0; n < THR_N; n++) {
            rB[n] = sB[n * DIM_Y + idy][k];
        }
        
        #pragma unroll
        for (n = 0; n < THR_N; n++) {
            #pragma unroll
            for (m = 0; m < THR_M; m++) {
                rC[n][m] += rA[m] * rB[n];
            }
        }
    }

    #pragma unroll
    for (n = 0; n < THR_N; n++) {
        int coord_dCn = bly * BLK_N + n * DIM_Y + idy;
        #pragma unroll
        for (m = 0; m < THR_M; m++) {
            int coord_dCm = blx * BLK_M + m * DIM_X + idx;
            if (coord_dCm < M && coord_dCn < N) {
                C[coord_dCn * M + coord_dCm] = rC[n][m];
            }
        }
    }
}

In [ ]:
import argparse
import math
import os
import time

import cupy as cp
import numpy as np

# main functions
def read_code(code_filename, params):
  ''' load kernel CUDA and set the params'''

  with open(code_filename, 'r') as f:
    code = f.read()
  for k, v in params.items():
    code = '#define ' + k + ' ' + str(v) + '\n' + code
  return code

def sgemm(A, B, dim_x=16, dim_y=16, blk_m=64, blk_n=64, blk_k=4, dim_xa=64, dim_ya=4, dim_xb=4, dim_yb=64):
  ''' sgemm kernel invocation'''

  assert A.dtype == cp.float32
  assert B.dtype == cp.float32
  assert(dim_x * dim_y == dim_xa * dim_ya == dim_xb * dim_yb)

  m, k = A.shape
  k, n = B.shape

  # Inputs matrices need to be in Fortran order.
  A = cp.asfortranarray(A)
  B = cp.asfortranarray(B)

  C = cp.empty((m, n), dtype=cp.float32, order='F')

  config = {'DIM_X': dim_x, 'DIM_Y': dim_y,
            'BLK_M': blk_m, 'BLK_N': blk_n, 'BLK_K': blk_k,
            'DIM_XA': dim_xa, 'DIM_YA': dim_ya,
            'DIM_XB': dim_xb, 'DIM_YB': dim_yb,
            'THR_M': blk_m // dim_x, 'THR_N': blk_n // dim_y}
  
  # load CUDA code
  code = read_code(sgemm_file, params=config)
  kern = cp.RawKernel(code, 'sgemm')

  # define the grid params and launch the kernel
  grid = (int(math.ceil(m / blk_m)), int(math.ceil(n / blk_n)), 1)
  block = (dim_x, dim_y, 1)
  args = (m, n, k, A, B, C)
  shared_mem = blk_k * (blk_m + 1) * 4 + blk_n * (blk_k + 1) * 4
  kern(grid, block, args=args, shared_mem=shared_mem)
  return C


In [ ]:
# load file CUDA sgemm.cu and exec the kernel sgemm

sgemm_file = 'sgemm.cu'
print('sgemm_file loaded : ', sgemm_file)

# define mat params and random instances
m = 20000
k = 20000
n = 20000
A = cp.random.uniform(low=-1., high=1., size=(m, k)).astype(cp.float32)
B = cp.random.uniform(low=-1., high=1., size=(k, n)).astype(cp.float32)

# test sgemm CUDA kernel and the cupy mat mult implementation
start = time.time()
sgemm(A, B)
cp.cuda.Stream.null.synchronize()
end = time.time()
print("\n      gemm kernel elapsed time: {} sec".format(end-start))
start = time.time()
cp.dot(A, B)
cp.cuda.Stream.null.synchronize()
end = time.time()
print('\ncupy matrix prod. elapsed time: {} sec'.format(end-start))
 

In [ ]:
!nvidia-smi


# FFT with pyTorch and Cupy

In [ ]:
%matplotlib inline
import torch
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
print('PyTorch: {}'.format(torch.__version__))
print('Numpy: {}'.format(np.__version__))
print('CuPy: {}'.format(cp.__version__))

In [ ]:
# Helper functions for each package to test FFT libs

def torch_ifft2_fft2(data):
    signal_ndim = 2
    data2 = torch.fft.fft(data, signal_ndim=signal_ndim)
    return torch.fft.ifft(data2, signal_ndim=signal_ndim)

def np_ifft2_fft2(data):
    data2 = np.fft.fft2(data)
    return np.fft.ifft2(data2)

def cp_ifft2_fft2(data):
    data2 = cp.fft.fft2(data)
    return cp.fft.ifft2(data2)


In [ ]:
sizes = [512, 1024, 2048, 4096] # X * X pixel arrays/tensors
names = []
totaltimes = []

name = 'PyTorch\nCPU'
device_type = 'cpu'
device = torch.device(device_type)
names.append(name)
torchtimes = []
datas = [torch.randn((x,x,2), device=device) for x in sizes]
print(f'Working on device: {datas[0].device}')
for x, data in zip(sizes, datas):
    print('{} {}x{}'.format(" ".join(name.split('\n')), x, x))
    t = %timeit -o torch_ifft2_fft2(data)
    torchtimes.append(t)
totaltimes.append(torchtimes)

In [ ]:
name = 'PyTorch\nGPU'
device_type = 'cuda'
device = torch.device('cuda:{}'.format(0))
names.append(name)
print('{} available: {}'.format(name, torch.cuda.is_available()))
torchtimes = []
datas = [torch.randn(size=(x,x,2), device=device) for x in sizes]
print(f'Working on device: {datas[0].device}')
for x, data in zip(sizes, datas):
    print('{} {}x{}'.format(" ".join(name.split('\n')), x, x))
    t = %timeit -o torch_ifft2_fft2(data)
    torchtimes.append(t)
totaltimes.append(torchtimes)

## Clear pytorch memory
del datas
torch.cuda.empty_cache()

In [ ]:
name = 'Numpy\nCPU'
names.append(name)
nptimes = []
datas = [np.random.normal(size=(x,x)).astype('complex128') for x in sizes]
for x, data in zip(sizes, datas):
    print('{} {}x{}'.format(" ".join(name.split('\n')), x, x))
    t = %timeit -o np_ifft2_fft2(data)
    nptimes.append(t)
totaltimes.append(nptimes)

In [ ]:
name = 'CuPy\nGPU'
names.append(name)
cptimes = []
device = cp.cuda.Device(0)
with device:
    datas = [cp.random.normal(size=(x,x)).astype('complex128') for x in sizes]
    for x, data in zip(sizes, datas):
        print('{} {}x{}'.format(" ".join(name.split('\n')), x, x))
        t = %timeit -o cp_ifft2_fft2(data)
        cptimes.append(t)
    totaltimes.append(cptimes)

# # Clear cupy memory
mempool = cp.get_default_memory_pool()
del datas
mempool.free_all_blocks()